In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
from typing import Any, Dict, List
import requests
import os

# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

# 알라딘 API 키 로드
aladin_api_key = os.getenv("ALADIN_API_KEY")



### 1. `import requests`
**`requests`**는 파이썬에서 **HTTP 요청(웹 통신)을 보내기 위해 가장 널리 사용되는 라이브러리**입니다.

*   **용도:** 웹 사이트의 데이터를 가져오거나(GET), 서버에 데이터를 보내거나(POST), 외부 API와 통신할 때 사용합니다.
*   **특징:** 파이썬 기본 라이브러리인 `urllib`보다 사용법이 훨씬 직관적이고 편리해서 "인간을 위한 HTTP 라이브러리"라고 불립니다.
*   **예시:**
    ```python
    import requests
    response = requests.get("https://api.github.com")
    print(response.status_code) # 200 (성공)
    ```

---

### 2. `from typing import Any, Dict, List, Union`
이것은 파이썬의 **`typing`**이라는 **내장 라이브러리**에서 가져온 것들입니다.

*   **용도:** **타입 힌트(Type Hinting)**를 작성할 때 사용합니다. 파이썬은 변수의 타입을 엄격하게 따지지 않지만, 코드를 읽는 사람이나 에디터(VS Code 등)에게 "이 변수에는 이런 데이터가 들어올 거야"라고 알려주는 역할을 합니다.
*   **각각의 의미:**
    *   **`Any`**: 어떤 타입이든 상관없음 (자유로운 타입)
    *   **`Dict`**: 딕셔너리 형태 (예: `Dict[str, int]` -> 키는 문자열, 값은 정수)
    *   **`List`**: 리스트 형태 (예: `List[str]` -> 문자열들로 이루어진 리스트)
    *   **`Union`**: "A이거나 B일 수 있음" (예: `Union[int, str]` -> 정수일 수도 있고 문자열일 수도 있음)

*   **예시:**
    ```python
    def process_data(data: List[str]) -> Dict[str, Any]:
        # data는 문자열 리스트여야 하고, 결과는 딕셔너리(키는 문자열, 값은 아무거나)를 반환한다.
        return {"result": data[0]}
    ```

---

### 요약
- **`requests`**: 웹 서버와 대화(통신)하기 위한 도구 (외부 설치 필요)
- **`typing`**: 코드를 더 명확하게 설명하기 위한 타입 표시 도구 (파이썬 기본 포함)

보통 LangChain이나 AI 에이전트 코드에서 `requests`는 외부 API(날씨, 검색 등)를 호출할 때 쓰고, `typing`은 복잡한 데이터 구조를 정의할 때 사용합니다.

In [ ]:
@tool
def get_aladin_bestsellers(count: int = 10) -> List[Dict[str, Any]]:
    """
    알라딘 API를 사용하여 현재 가장 인기 있는 베스트셀러 도서 목록을 가져옵니다.
    
    Args:
        count (int): 가져올 도서의 수 (기본값 10, 최대 50).
        
    Returns:
        List[Dict]: 도서 제목, 저자, 가격, 링크 등을 포함한 딕셔너리 리스트.
    """
    # 2025-04-07 지침: 보안을 위해 TTBKey는 환경변수 사용 권장
    url = "http://www.aladin.co.kr/ttb/api/ItemList.aspx"
    
    # 알라딘 API 명세서 근거 파라미터 구성
    params = {
        "ttbkey": aladin_api_key,
        "QueryType": "Bestseller",  # 리스트 종류: 베스트셀러
        "MaxResults": min(count, 50), # 한 페이지 최대 50개 제한
        "start": 1,
        "SearchTarget": "Book",
        "output": "js",            # JSON 방식 출력
        "Version": "20131101"      # 최신 버전
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status() 
        # 요청 결과가 4xx 또는 5xx 에러(실패)일 때 예외(requests.exceptions.HTTPError)를 발생시킵니다.
        # 즉, 요청이 성공(200 OK 등)이 아니면 코드 실행을 중단하고 에러 처리를 하도록 만듭니다.
        # 이를 통해 API 호출 실패를 쉽게 감지하고, try-except로 예외를 처리할 수 있습니다.
        data = response.json()
        
        # 문서 응답 구조: 'item' 키 안에 도서 리스트가 담겨 있음
        items = data.get("item", [])
        
        # 에이전트가 이해하기 쉽도록 핵심 필드만 필터링 (Token 절약 및 가독성)
        return [
            {
                "title": item.get("title"),
                "author": item.get("author"),
                "publisher": item.get("publisher"),
                "price": item.get("priceSales"),
                "link": item.get("link")
            }
            for item in items
        ]
    except Exception as e:
        return [{"error": f"API 호출 실패: {str(e)}"}]

In [13]:
from pprint import pprint

agent = create_agent(
    model=model,
    tools=[get_aladin_bestsellers],)


result = agent.invoke({"messages": [
  {"role": "user", "content": "현재 알라딘에서 가장 인기 있는 베스트셀러 책 5권을 알려줘."},
]})

print("Agent의 응답:", result['messages'][-1].content) 



Agent의 응답: 현재 알라딘에서 가장 인기 있는 베스트셀러 책 5권은 다음과 같습니다:

1. **[프로젝트 헤일메리 (영화 특별판)](https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=385526244&amp;partner=openAPI&amp;start=api)**
   - 저자: 앤디 위어 (지은이), 강동혁 (옮긴이)
   - 출판사: 알에이치코리아(RHK)
   - 가격: 19,800원

2. **[괴테는 모든 것을 말했다 - 제172회 아쿠타가와상 수상작](https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=376765918&amp;partner=openAPI&amp;start=api)**
   - 저자: 스즈키 유이 (지은이), 이지수 (옮긴이)
   - 출판사: 리프
   - 가격: 15,300원

3. **[인생을 위한 최소한의 생각](https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=385481121&amp;partner=openAPI&amp;start=api)**
   - 저자: 신영준, 고영성 (지은이)
   - 출판사: 상상스퀘어
   - 가격: 17,820원

4. **[완벽한 원시인 - 10만 년을 되돌려 되찾는 뇌 설계도](https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=386947012&amp;partner=openAPI&amp;start=api)**
   - 저자: 자청 (지은이)
   - 출판사: 필로틱
   - 가격: 19,800원

5. **[엄마와 딸들의 미친년의 역사](https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=388054879&amp;partner=openAPI&amp;start=api)**
   - 저자: 이랑 (지은이)
   - 출판사: 이야기장수
   - 가격: 16,020원

관심 있는 책을 클릭하

create_agent로 생성된 에이전트 환경에서 도구(tool)가 Exception을 발생시키지 않고, 코드처럼 에러 메시지가 담긴 리스트를 반환하게 되면 다음과 같은 흐름으로 응답하게 됩니다.

1. 에이전트의 인식 과정
에이전트는 이 상황을 "프로그램이 멈춘 에러"가 아니라, **"도구를 실행했더니 결과값으로 에러 메시지가 돌아왔다"**고 인식합니다.

에이전트가 get_aladin_bestsellers 도구를 호출합니다.
API 호출 중 문제가 생겨 except 블록이 실행됩니다.
도구는 [{"error": "API 호출 실패: ..."}]라는 데이터를 반환합니다.
에이전트는 이 텍스트 데이터를 도구의 정상적인 실행 결과로 받아봅니다.
2. 에이전트의 최종 응답 (사용자가 보게 될 내용)
에이전트는 도구로부터 받은 에러 정보를 바탕으로 사용자에게 상황을 설명하는 답변을 생성합니다. 보통 다음과 같이 친절하게 응답합니다.

에이전트의 예상 응답: "죄송합니다. 현재 알라딘 API 서버와의 통신에 문제가 발생하여 베스트셀러 정보를 가져오지 못했습니다. (에러 내용: API 호출 실패: ...)"

3. 왜 이렇게 처리하나요? (장점)
만약 except에서 데이터를 반환하지 않고 그냥 에러를 던져버리면(raise), 전체 파이썬 프로그램이 멈춰버립니다.

하지만 위 코드처럼 에러 내용을 텍스트로 반환하면:

프로그램이 죽지 않고 계속 실행됩니다.
에이전트(LLM)가 "아, 지금 API가 안 되는구나"라고 판단하여 사용자에게 대안을 제시하거나 사과를 할 수 있는 기회를 갖게 됩니다.
요약
사용자는 "시스템 에러 화면"을 보는 대신, **"AI가 API 호출에 실패했다고 설명해주는 답변"**을 받게 됩니다. 이것이 에이전트 설계에서 매우 중요한 예외 처리(Error Handling) 방식입니다.

**사용자가 말한 "5권"이라는 정보가 도구(`get_aladin_bestsellers`)의 `count` 인자(Argument)로 전달됩니다.**

이것이 바로 **LLM 에이전트의 핵심 기능**인 **"함수 호출(Function Calling)"** 과정입니다. 단계별로 어떻게 일어나는지 설명해 드릴게요.

### 1. LLM의 분석 (추론)
사용자가 "5권을 알려줘"라고 말하면, LLM은 자기가 가진 도구 목록을 살펴봅니다.
- 도구 이름: `get_aladin_bestsellers`
- 입력 인자: `count` (기본값 10)

LLM은 사용자의 의도를 분석하여 **"아, `get_aladin_bestsellers` 도구를 써야겠는데, `count`에는 5를 넣어야겠구나!"**라고 판단합니다.

### 2. 도구 호출 (실행)
에이전트는 LLM의 판단에 따라 실제로 다음과 같이 함수를 호출합니다.
```python
# LLM이 판단한 결과에 따라 에이전트가 내부적으로 실행함
get_aladin_bestsellers(count=5)
```
이때 `count: int = 10`으로 되어 있던 기본값은 무시되고, 사용자가 요청한 **5**가 우선적으로 적용됩니다.

### 3. 만약 권수를 말하지 않았다면?
사용자가 그냥 "베스트셀러 알려줘"라고만 했다면, LLM은 `count` 값을 특정하지 않을 수도 있습니다. 그럴 때는 함수 정의에 있는 **기본값인 10**이 사용됩니다.

### 4. 왜 이게 가능한가요?
함수 정의 위에 붙어 있는 **Docstring(주석)**이나 **타입 힌트** 덕분입니다.
```python
def get_aladin_bestsellers(count: int = 10) -> List[Dict[str, Any]]:
    """알라딘 베스트셀러 목록을 가져옵니다. (count: 가져올 책의 권수)"""
```
LLM은 이 설명을 읽고 "아, `count`라는 숫자를 넣으면 그만큼 가져오는구나!"라고 학습된 지식을 바탕으로 이해하는 것입니다.

### 요약
- **네, 사용자의 "5권" 요청은 도구의 `count` 값으로 연결됩니다.**
- LLM이 똑똑하게 사용자의 문장에서 숫자를 추출해 함수 인자로 매칭시켜 줍니다.